# 04 · Función de contribución

Notebook de demostración de la parte 04 (`code/reward.py`). Guía: `docs/instrucciones/04_contribucion.md`. Todo usa el
**train** de cada fold; ningún agente se entrena aquí.

**Contribución elegida (terminal, pagada al cierre de la operación):**

$$C(S, x, W) = \ln\Big(1 + s\,\frac{P_{sal} - P_{ent}}{P_{ent}} - c\,\frac{P_{ent} + P_{sal}}{P_{ent}}\Big),\qquad s \in \{+1, -1\},\; c = 0.00125,\qquad C(S, \text{esperar}, W) = 0$$

Con todo el capital en cada operación, $\sum C = \ln(\text{capital final})$: maximizar la suma esperada de contribuciones es
maximizar el crecimiento log del capital.

In [1]:
import sys
sys.path.insert(0, "..")
sys.modules.pop("code", None)  # IPython ya importó el módulo `code` estándar (ver docs/decisiones.md)

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from code import data, env, reward
from main import BTC_CSV, FOLDS, SEED

ohlcv, _ = data.load_ohlcv(BTC_CSV)
features = data.compute_features(ohlcv)
folds = {k: data.build_fold_dataset(ohlcv, features, k, FOLDS[k]) for k in FOLDS}

## 1. Una operación a mano

Largo a 100, sale por stop en 98: paga el costo de entrada **y** el de salida.

In [2]:
entry, exit_ = 100.0, 98.0
gross = (exit_ - entry) / entry
cost = env.COST_PER_SIDE * (entry + exit_) / entry
trade = env.TradeOutcome(1, 3.0, gross, cost, gross - cost, np.log1p(gross - cost), 0.01, "stop")
pd.Series({name: fn(trade) for name, fn in reward.REWARDS.items()} | {"a mano": np.log(1 - 0.02 - 0.00125 * 1.98)}).round(6)

net_log_return      -0.022731
vol_scaled_return   -0.464003
gross_log_return    -0.020203
a mano              -0.022731
dtype: float64

## 2. Las tres contribuciones sobre las mismas operaciones (train del fold 1)

Se graba cada `TradeOutcome` de un recorrido completo del train y se le aplica cada contribución. Con costos, el aleatorio
pierde (criterio de terminado); la bruta casi no distingue operar al azar de no operar.

In [3]:
comparison = reward.reward_comparison(folds[1], "train", seed=SEED)
comparison[["trades", "mean_per_trade", "total"]].unstack("reward").round(4)

trades                                   \
reward        gross_log_return net_log_return vol_scaled_return   
policy                                                            
aleatorio                 2167           2167              2167   
siempre_corto             2254           2254              2254   
siempre_largo             2254           2254              2254   

                mean_per_trade                                   \
reward        gross_log_return net_log_return vol_scaled_return   
policy                                                            
aleatorio              -0.0002        -0.0027           -0.0849   
siempre_corto          -0.0016        -0.0041           -0.1266   
siempre_largo           0.0002        -0.0023           -0.0723   

                         total                                   
reward        gross_log_return net_log_return vol_scaled_return  
policy                                                           
aleatorio              -0.4298        -5.8615         -184.0577  
siempre_corto          -3.5773        -9.2383         -285.4420  
siempre_largo           0.5295        -5.1137         -163.0111

**Aleatorio en los 4 folds:** contribución media por decisión, neta y bruta.

In [4]:
by_fold = {k: reward.reward_comparison(folds[k], "train", seed=SEED).loc["aleatorio", "mean_per_decision"] for k in FOLDS}
pd.DataFrame(by_fold).T.rename_axis("fold").map(lambda v: f"{v:.2e}")

reward,net_log_return,vol_scaled_return,gross_log_return
fold,,,
1,-1.80e-03,-5.66e-02,-1.32e-04
2,-1.97e-03,-6.24e-02,-3.06e-04
3,-1.86e-03,-6.60e-02,-2.04e-04
4,-1.75e-03,-6.59e-02,-8.37e-05


## 3. Escala: log neto (A) contra escalado por volatilidad (B)

A queda en el orden de 10⁻², como piden las notas de clase. B queda en el orden de 1, con la misma escala en todas las
épocas, pero su suma ya no es el capital.

In [5]:
outcomes = [o for o in reward.collect_outcomes(folds[1], "train", "aleatorio", SEED) if o.side != 0]
a = np.array([reward.net_log_return(o) for o in outcomes])
b = np.array([reward.vol_scaled_return(o) for o in outcomes])
fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))
axes[0].hist(a, bins=80, color="#257de0"); axes[0].set(title="A: log neto por operación", xlabel="log del capital")
axes[1].hist(b, bins=80, color="#f47230"); axes[1].set(title="B: log neto / (σ̂ √H)", xlabel="unidades de σ")
plt.tight_layout()
pd.DataFrame({"A": pd.Series(a).describe(), "B": pd.Series(b).describe()}).round(4)

,A,B
count,2167.0000,2167.0000
mean,-0.0027,-0.0849
std,0.0354,0.7980
min,-0.1987,-1.3956
25%,-0.0246,-1.0773
50%,-0.0015,-0.0533
75%,0.0203,0.8530
max,0.1942,0.9344


**Escala de A por año** (siempre largo, train del fold 4): la desviación de una operación cambia con el régimen; es el
costo de no escalar.

In [6]:
e = env.make_env(folds[4], "train", reward.net_log_return, mode="eval")
e.reset(seed=SEED)
done = False
while not done:
    _, _, term, trunc, _ = e.step(env.LONG)
    done = term or trunc
log = e.trade_log()
log["contribution"] = np.log(log["equity_after"] / log["equity_before"])
log.groupby(log["decision_time"].dt.year)["contribution"].agg(["count", "mean", "std"]).round(4)

,count,mean,std
decision_time,,,
2017,157,0.0054,0.0573
2018,507,-0.0058,0.0399
2019,530,-0.0020,0.0274
2020,529,-0.0000,0.0284
2021,531,-0.0036,0.0375
2022,528,-0.0042,0.0275
2023,541,-0.0006,0.0166
2024,543,-0.0020,0.0219


## 4. Terminal contra por barra (pregunta 3)

La misma operación pagada por barra suma lo mismo (se reparte el log neto), pero **cambia cuándo se paga**:

- el costo de entrada llega como castigo inmediato en la primera barra y el de salida al final;
- con descuento por barra, la ganancia tardía pesa menos que la temprana, aunque el agente no decide nada en medio
  (el gestor cierra la posición): la recompensa de esas barras no depende de ninguna acción que pueda cambiar.

In [7]:
parts = reward.per_bar_contributions(100.0, [101.0, 99.0, 103.0], 102.0, side=1)
gamma = 0.99  # por hora
pd.DataFrame({"por barra": parts, "peso γ^k": gamma ** np.arange(len(parts))}).assign(
    descontada=lambda d: d["por barra"] * d["peso γ^k"]).round(5)

,por barra,peso γ^k,descontada
0,0.00871,1.0000,0.00871
1,-0.02003,0.9900,-0.01983
2,0.03966,0.9801,0.03887
3,-0.01102,0.9703,-0.01069


**Con las operaciones reales** (siempre largo, train del fold 1): relación señal-ruido de una recompensa por barra
contra la de una por operación. Por barra, el promedio es casi cero frente a su ruido: el crédito de abrir la posición
se diluye en ≈ 24 recompensas que reflejan sobre todo el ruido de cada hora.

In [8]:
e = env.make_env(folds[1], "train", reward.net_log_return, mode="eval")
e.reset(seed=SEED)
done = False
while not done:
    _, _, term, trunc, _ = e.step(env.LONG)
    done = term or trunc
trades = e.trade_log()
close = folds[1].ohlcv["train"]["close"]
per_bar, terminal = [], []
for t in trades.itertuples():
    closes = close.loc[t.entry_time:t.exit_time].iloc[:-1].to_numpy()
    side = 1 if t.side == "largo" else -1
    parts = reward.per_bar_contributions(t.entry_price, closes, t.exit_price, side)
    per_bar.extend(parts)
    terminal.append(parts.sum())
per_bar, terminal = np.array(per_bar), np.array(terminal)
assert np.allclose(terminal, np.log(trades["equity_after"] / trades["equity_before"]))
pd.DataFrame({
    "recompensas": [len(per_bar), len(terminal)],
    "media": [per_bar.mean(), terminal.mean()],
    "desviación": [per_bar.std(), terminal.std()],
    "media / desviación": [per_bar.mean() / per_bar.std(), terminal.mean() / terminal.std()],
}, index=["por barra", "terminal"]).round(5)

,recompensas,media,desviación,media / desviación
por barra,37501,-0.00014,0.00906,-0.01504
terminal,2254,-0.00227,0.03590,-0.06319


## 5. Modos de falla (trampas de §3.4)

| Trampa | Cómo la trata esta contribución |
|---|---|
| P&L bruto premia el riesgo apalancado | Se usa el **log** neto con todo el capital y sin apalancamiento: una pérdida de 50 % cuesta −0.69 y una ganancia de 50 % da +0.41; la concavidad castiga apostar fuerte. El P&L bruto (C) queda descartado: no cobra costos. |
| Sharpe por episodio no es markoviano | No se usa: cada contribución depende solo de la operación que cierra, no de la historia del episodio. |
| Log por paso ignora el drawdown | La suma de logs **es** el capital final, así que una caída se paga completa; lo que no castiga es la **trayectoria** (dos caminos al mismo capital valen igual). El estado sí ve la caída desde el máximo (`equity_drawdown`). Se acepta y se mide el drawdown en la evaluación (parte 06). |
| Coeficientes de moldeo son hiperparámetros | No hay ninguno: esperar vale 0 y no hay penalización de inventario. El único parámetro, γ, es del algoritmo (parte 05). |